# Isolating stability

Control the spectral radius of Parcae's diagonal injection and measure how the
recurrent dynamics, stages of inference, and test-time perplexity change. The
140M sweep crosses six radius settings with fixed-depth and clipped-Poisson
training. The learned arm learns a per-channel decay instead of fixing it to one rho.

Every figure retains the complete radius order. Quick mode collects 0.25, 0.75,
1.25, and learned under both training schemes; omitted panels say **skipped**.
Full mode collects all twelve arms. Both modes use a 128-recurrence analysis
and the same perplexity sweep. Quick saves only to its run directory; full also
refreshes paper figures. Explicit sample limits do not change that export policy.

In [ ]:
mode = "quick"
dataset = "gsm8k"  # "gsm8k" or "hellaswag"
arms = None  # Quick: 0.25, 1.0, learned; full: all five rhos, both schemes.
analysis_recurrences = 128  # Shared by convergence, stages, and norm growth.
ppl_recurrences = [1, 2, 4, 8, 16, 32, 64]
training_recurrence = 8
max_training_recurrence = 16  # Upper clipping bound for Poisson training.
limit = None  # Analysis examples: 1 (quick), 256 (full); 0 uses the whole split.
ppl_limit = None  # Perplexity examples: 1 (quick), 32 (full), as in the original sweep.
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import gc
import hashlib
import math
from contextlib import nullcontext
from dataclasses import asdict
from itertools import pairwise

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from experiments.cache import ExperimentCache
from experiments.support import (
    COLORMAPS,
    STABILITY,
    _set_seed,
    add_depth_colorbar,
    add_prelude_coda_connectors,
    add_prelude_coda_legend,
    configure_experiment,
    create_run_dir,
    export_paper_figures,
    get_figsize,
    load_experiment_model,
    model_subplots,
    sample_indices,
    save_figure,
    write_tokenization,
)
from looped_internals.analysis import compute_attn_pattern_diff_norm, get_mixing_scores
from looped_internals.capture import compute_model_run_data, metrics_names_filter
from looped_internals.evaluation.lm_eval import get_task_manager
from looped_internals.models.tokenization import tokenizer_metadata

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# Full figure positions and colours stay fixed even when collection uses a subset.
RHO_KEYS = ("0_25", "0_5", "0_75", "1_0", "learned")
RHOS = tuple(rho.replace("_", ".") for rho in RHO_KEYS)
SCHEDULES = ("poisson", "fixed")
SUMMARY_ARMS = ("stab_poisson_0_25", "stab_poisson_1_0")
QUICK_RHOS = ("0_25", "1_0", "learned")
figure_arms = {
    schedule: [f"stab_{schedule}_{rho}" for rho in RHO_KEYS] for schedule in SCHEDULES
}
if arms is None:
    selected_rhos = RHO_KEYS if mode == "full" else QUICK_RHOS
    arms = [f"stab_{schedule}_{rho}" for schedule in SCHEDULES for rho in selected_rhos]
else:
    arms = list(arms)
if not arms or len(set(arms)) != len(arms) or set(arms) - STABILITY.keys():
    raise ValueError("Select registered stability arms without duplicates")
if not ppl_recurrences or any(type(t) is not int or t < 1 for t in ppl_recurrences):
    raise ValueError("ppl_recurrences must contain positive integers")
ppl_recurrences = sorted(set(ppl_recurrences))
if training_recurrence not in ppl_recurrences:
    raise ValueError("ppl_recurrences must include training_recurrence")
if max_training_recurrence < training_recurrence:
    raise ValueError("max_training_recurrence must reach training_recurrence")
if ppl_limit is not None and (type(ppl_limit) is not int or ppl_limit < 1):
    raise ValueError("ppl_limit must be a positive integer or None")

sample_limit = limit if limit is not None else (1 if mode == "quick" else 256)
ppl_examples = ppl_limit if ppl_limit is not None else (1 if mode == "quick" else 32)
config = configure_experiment(
    experiment="isolating_stability",
    dataset=dataset,
    mode=mode,
    models=arms,
    num_steps=analysis_recurrences,
    limit=sample_limit,
    seed=seed,
    device=device,
    dtype=dtype,
    output_dir=output_dir,
    compute_residual_entropy=True,
)
if (
    any(name.startswith("stab_poisson_") for name in arms)
    and config.num_steps < max_training_recurrence
):
    raise ValueError(
        "analysis_recurrences must reach the Poisson training-max reference"
    )
# Only these pairs run. An overlapping PPL/analysis pair shares its forward pass.
analysis_pairs = [(name, analysis_recurrences) for name in arms]
ppl_pairs = [(name, steps) for name in arms for steps in ppl_recurrences]
control_pairs = [
    (name, max_training_recurrence) for name in SUMMARY_ARMS if name in arms
]
model_recurrences = list(dict.fromkeys([*ppl_pairs, *analysis_pairs, *control_pairs]))


def arm_parts(name):
    _, schedule, rho = name.split("_", 2)
    return schedule, rho.replace("_", ".")


result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
ATTENTION_METRICS = {
    "sink_rates": "Sink rate",
    "mixing_scores": "Mixing score",
    "colsum_concentrations": "Column concentration",
}
RESIDUAL_METRICS = {
    "residual_norms_mean": "Mean residual norm",
    "residual_entropy": "Residual entropy",
}
STAGE_METRICS = ATTENTION_METRICS | RESIDUAL_METRICS
CONVERGENCE_METRICS = {
    "to_final": "Distance to final state",
    "successive": "Successive difference norm",
    "attention_to_training_max": "Attn. Frobenius distance",
}


def evaluation_inputs(bridge, indices, task):
    """Preserve the sweep's native encoding of context plus reference continuation."""
    inputs = {}
    for index in indices:
        doc = task.eval_docs[index]
        text = task.doc_to_text(doc) + str(task.doc_to_target(doc))
        ids = bridge.tokenizer.encode(text, return_tensors="pt")
        if ids.shape[1] > 1:
            inputs[index] = ids.cpu()
    return inputs


def summarize_convergence(cache, layout, *, attention_reference=None):
    """Same-layer residual distances and optional attention distance to training depth."""
    layers = sorted(
        {layer.layer_index for layer in layout if layer.iteration is not None}
    )
    curves = {"to_final": [], "successive": []}
    if attention_reference is not None:
        curves["attention_to_training_max"] = []
    for layer_index in layers:
        applications = [layer for layer in layout if layer.layer_index == layer_index]
        states = [cache[layer.residual_key] for layer in applications]
        final = states[-1].float()
        curves["to_final"].append(
            [
                torch.linalg.vector_norm(state.float() - final, dim=-1).mean().item()
                for state in states
            ]
        )
        curves["successive"].append(
            [
                torch.linalg.vector_norm(after.float() - before.float(), dim=-1)
                .mean()
                .item()
                for before, after in pairwise(states)
            ]
        )
        if attention_reference is not None:
            reference_layer = next(
                layer
                for layer in applications
                if layer.iteration == attention_reference - 1
            )
            reference = cache[reference_layer.attn_key].float()
            curves["attention_to_training_max"].append(
                [
                    compute_attn_pattern_diff_norm(cache[layer.attn_key], reference)
                    .mean()
                    .item()
                    for layer in applications
                ]
            )
    return curves


def summarize_analysis(cache, ids, structure, num_steps, *, attention_reference=None):
    """Reduce the same forward pass into stage profiles and convergence curves."""
    layout = list(structure.virtual_layers(num_steps))
    run = compute_model_run_data(
        cache,
        ids,
        structure,
        compute_pairwise=False,
        compute_differences=False,
        compute_residual_entropy=True,
    )[0]
    curves = {}
    for field in STAGE_METRICS:
        values = run.get_metric(field).float()
        if values.ndim > 1:
            values = values.mean(dim=-1)  # Average attention heads within each example.
        curves[field] = values.cpu().tolist()
    return {
        "layers": [layer.layer_index for layer in layout],
        "virtual_layers": [layer.virtual_layer_index for layer in layout],
        "recurrences": [layer.iteration for layer in layout],
        "recurrent_layers": sorted(
            {layer.layer_index for layer in layout if layer.iteration is not None}
        ),
        "curves": curves,
        "convergence": summarize_convergence(
            cache, layout, attention_reference=attention_reference
        ),
    }


def aggregate_examples(examples):
    """Average examples equally; bands use one sample standard deviation."""
    data = {
        key: examples[0][key]
        for key in ("layers", "virtual_layers", "recurrences", "recurrent_layers")
    }
    for group in ("curves", "convergence"):
        data[group] = {}
        for metric in examples[0][group]:
            values = np.asarray([example[group][metric] for example in examples])
            data[group][metric] = {
                "mean": values.mean(axis=0),
                "std": values.std(axis=0, ddof=1 if len(examples) > 1 else 0),
            }
    return data


def recurrent_norm_by_recurrence(examples):
    """Average token-mean norms over core layers, then average examples equally."""
    recurrences = np.asarray(examples[0]["recurrences"])
    steps = sorted({r for r in recurrences if r is not None})
    values = np.asarray(
        [example["curves"]["residual_norms_mean"] for example in examples]
    )
    per_example = np.stack(
        [values[:, recurrences == r].mean(axis=1) for r in steps], axis=1
    )
    return np.asarray(steps) + 1, per_example.mean(axis=0)


def summarize_training_control(cache, structure, num_steps):
    """Keep prelude, the final recurrence, and the actual training-depth coda."""
    layout = [
        layer
        for layer in structure.virtual_layers(num_steps)
        if layer.iteration is None or layer.iteration == num_steps - 1
    ]
    return {
        "layers": [layer.layer_index for layer in layout],
        "virtual_layers": [layer.virtual_layer_index for layer in layout],
        "recurrences": [layer.iteration for layer in layout],
        "recurrent_layers": sorted(
            {layer.layer_index for layer in layout if layer.iteration is not None}
        ),
        "curves": {
            "mixing_scores": [
                get_mixing_scores(cache[layer.attn_key].float()).mean().item()
                for layer in layout
            ]
        },
        "convergence": {},
    }

## Generate and reduce

Load each selected checkpoint once. The perplexity sweep and the 128-recurrence
analysis and the two training-depth mixing controls reuse that load. The analysis collects convergence, attention, residual
norm, and residual entropy together; no extra rollout is needed for each figure.
If an analysis length is also in the PPL sweep, shared examples reuse one forward.

Both sample selections use the same seeded shuffle. As in the original sweep,
inputs are the concatenated likelihood-task context and reference continuation,
encoded using the checkpoint's native tokenizer. PPL scores every next token,
including context tokens. The seed is reset before each forward pass so changes
in rollout length do not introduce a fresh initial-state draw for the same input.
This experiment does not estimate uncertainty over initialization seeds.

The two summary controls collect attention only at the training maximum, using
all analysis examples and sharing forwards with PPL where possible. Their compact
mixing-score summaries have separate cache entries, preserving the existing
analysis cache keys. PPL-only passes collect no activations. Analysis caches live on the CPU and are
reduced and released after each example. Poisson arms additionally compare each
recurrent layer's attention with its own pattern at the configured training maximum,
using that same analysis cache. Other attention comparisons and all-pairs metrics
are disabled. Per-example summaries, corpus NLL/token totals, input
hashes, and recorded layer layouts are saved for reproducibility. Each completed
checkpoint is saved separately before the final overview. Rerunning figure cells
uses those compact results without loading models or repeating inference.

In [ ]:
run_dir = create_run_dir(
    config,
    cache=result_cache.settings,
    arms=arms,
    figure_arms=figure_arms,
    model_recurrences=model_recurrences,
    analysis_pairs=analysis_pairs,
    ppl_pairs=ppl_pairs,
    control_pairs=control_pairs,
    ppl_examples=ppl_examples,
    training_recurrence=training_recurrence,
    max_training_recurrence=max_training_recurrence,
    input_protocol="native_context_plus_reference_all_next_tokens",
    convergence_reference="same_physical_layer_at_final_analysis_recurrence",
    attention_reference="same_physical_layer_at_max_training_recurrence_in_analysis_run",
)
for folder in ("models", "convergence", "recurrent"):
    (run_dir / folder).mkdir()

task = get_task_manager().load([config.task])["tasks"][config.task]
split_size = len(task.eval_docs)
analysis_indices = sorted(sample_indices(split_size, config.limit, seed))
ppl_indices = sorted(sample_indices(split_size, ppl_examples, seed))
summaries, curves, run_info, training_controls = {}, {}, {}, {}


def collect_stability_arm(name, analysis_steps, entry, control_entry):
    collect_analysis = not entry.complete
    collect_control = control_entry is not None and not control_entry.complete
    print(
        f"Collecting {name}: analysis={collect_analysis}, training control={collect_control}",
        flush=True,
    )
    _set_seed(seed)
    bridge, structure = load_experiment_model(
        name,
        config,
        steps=analysis_steps,
        **({"cache": result_cache} if result_cache.enabled else {}),
    )
    try:
        attention_reference = (
            max_training_recurrence if arm_parts(name)[0] == "poisson" else None
        )
        indices = sorted(set(analysis_indices) | set(ppl_indices))
        inputs = evaluation_inputs(bridge, indices, task)
        selected_analysis = [i for i in analysis_indices if i in inputs]
        selected_ppl = [i for i in ppl_indices if i in inputs]
        if not selected_analysis or not selected_ppl:
            raise ValueError(f"No usable evaluation inputs for {name}")
        examples = []
        progress = entry.progress
        contributions = progress["value"] if progress else {}
        control_progress = control_entry.progress if collect_control else None
        control_examples = control_progress["value"] if control_progress else {}
        nll_totals = {}
        steps_to_run = (
            set(ppl_recurrences) | {analysis_steps} if collect_analysis else set()
        )
        if collect_control:
            steps_to_run.add(max_training_recurrence)
        for steps in sorted(steps_to_run):
            measure_analysis = collect_analysis and steps == analysis_steps
            measure_ppl = collect_analysis and steps in ppl_recurrences
            measure_control = collect_control and steps == max_training_recurrence
            selected = sorted(
                (
                    set(selected_analysis)
                    if measure_analysis or measure_control
                    else set()
                )
                | (set(selected_ppl) if measure_ppl else set())
            )
            total_nll, total_tokens = 0.0, 0
            for index in selected:
                collect = measure_analysis and index in selected_analysis
                score = measure_ppl and index in selected_ppl
                missing_analysis = (collect or score) and (
                    steps,
                    index,
                ) not in contributions
                missing_control = (
                    measure_control
                    and index in selected_analysis
                    and index not in control_examples
                )
                if missing_analysis or missing_control:
                    ids = inputs[index].to(config.device)
                    _set_seed(seed)
                    with torch.no_grad():
                        capture_analysis = collect and missing_analysis
                        if capture_analysis or missing_control:
                            logits, cache = bridge.run_with_cache(
                                ids,
                                accumulate=True,
                                names_filter=metrics_names_filter
                                if capture_analysis
                                else lambda name: name.endswith(".attn.hook_pattern"),
                                device="cpu",
                                num_steps=steps,
                            )
                        else:
                            logits = bridge(ids, return_type="logits", num_steps=steps)
                        if missing_analysis:
                            nll, tokens = 0.0, 0
                            if score:
                                nll = F.cross_entropy(
                                    logits[0, :-1].float(), ids[0, 1:], reduction="sum"
                                ).item()
                                tokens = ids.shape[1] - 1
                            analysis = (
                                summarize_analysis(
                                    cache,
                                    ids,
                                    structure,
                                    steps,
                                    attention_reference=attention_reference,
                                )
                                if capture_analysis
                                else None
                            )
                            contributions[(steps, index)] = (analysis, nll, tokens)
                            entry.checkpoint(contributions)
                        if missing_control:
                            control_examples[index] = summarize_training_control(
                                cache, structure, steps
                            )
                            control_entry.checkpoint(control_examples)
                        del logits
                        if capture_analysis or missing_control:
                            del cache
                    del ids
                if collect or score:
                    analysis, nll, tokens = contributions[(steps, index)]
                    total_nll += nll
                    total_tokens += tokens
                    if collect:
                        examples.append(analysis)
            if measure_ppl:
                nll_totals[steps] = {
                    "total_nll": total_nll,
                    "total_tokens": total_tokens,
                }
        perplexity = {
            steps: math.exp(totals["total_nll"] / totals["total_tokens"])
            for steps, totals in nll_totals.items()
        }
        info = {
            "analysis_steps": analysis_steps,
            "attention_reference_recurrence": attention_reference,
            "analysis_indices": selected_analysis,
            "ppl_indices": selected_ppl,
            "requested_analysis_indices": analysis_indices,
            "requested_ppl_indices": ppl_indices,
            "split_size": split_size,
            "nll_totals": nll_totals,
            "layer_groups": [asdict(group) for group in structure.groups],
            "virtual_layers": [
                asdict(layer) for layer in structure.virtual_layers(analysis_steps)
            ],
            "inputs": [
                {
                    "dataset_index": index,
                    "length": ids.numel(),
                    "first_ids": ids.flatten()[:8].tolist(),
                    "sha256": hashlib.sha256(ids.numpy().tobytes()).hexdigest(),
                }
                for index, ids in inputs.items()
            ],
        }
        result = {
            "summaries": examples,
            "perplexity": perplexity,
            "run": info,
            "tokenization": {
                "bos_policy": "native",
                **tokenizer_metadata(bridge.tokenizer),
            },
        }
        if not collect_analysis:
            result = entry.load()
        control = None
        if control_entry is not None:
            control = (
                {
                    "summaries": [
                        control_examples[index] for index in selected_analysis
                    ],
                    "run": {
                        "num_steps": max_training_recurrence,
                        "indices": selected_analysis,
                        "inputs": [
                            item
                            for item in info["inputs"]
                            if item["dataset_index"] in selected_analysis
                        ],
                    },
                    "tokenization": result["tokenization"],
                }
                if collect_control
                else control_entry.load()
            )
        return result, control
    finally:
        del bridge, structure
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


for name, analysis_steps in analysis_pairs:
    key = result_cache.key(
        config,
        name,
        analysis_version=analysis_version,
        task=config.task,
        analysis_indices=analysis_indices,
        ppl_indices=ppl_indices,
        texts=[
            (task.doc_to_text(task.eval_docs[i]), task.doc_to_target(task.eval_docs[i]))
            for i in sorted(set(analysis_indices) | set(ppl_indices))
        ],
        ppl_recurrences=ppl_recurrences,
        max_training_recurrence=max_training_recurrence,
    )
    control_context = nullcontext(None)
    if name in SUMMARY_ARMS:
        control_key = result_cache.key(
            config,
            name,
            analysis_version=analysis_version,
            num_steps=max_training_recurrence,
            task=config.task,
            indices=analysis_indices,
            texts=[
                (
                    task.doc_to_text(task.eval_docs[i]),
                    task.doc_to_target(task.eval_docs[i]),
                )
                for i in analysis_indices
            ],
            input_protocol="native_context_plus_reference_all_next_tokens",
        )
        control_context = result_cache.entry("isolating_stability_control", control_key)
    with (
        result_cache.entry("isolating_stability", key) as entry,
        control_context as control_entry,
    ):
        if entry.complete and (control_entry is None or control_entry.complete):
            result = entry.load()
            control = control_entry.load() if control_entry is not None else None
        else:
            result, control = collect_stability_arm(
                name, analysis_steps, entry, control_entry
            )
        if not entry.complete:
            entry.save(result)
        if control_entry is not None:
            if not control_entry.complete:
                control_entry.save(control)
            training_controls[name] = control
            control_entry.export(
                control, run_dir / "models" / f"{name}-training-control.pt"
            )
        summaries[name], curves[name], run_info[name] = (
            result["summaries"],
            result["perplexity"],
            result["run"],
        )
        write_tokenization(run_dir, name, result["tokenization"])
        entry.export(result, run_dir / "models" / f"{name}.pt")

stages = {name: aggregate_examples(examples) for name, examples in summaries.items()}
controls = {
    name: aggregate_examples(control["summaries"])
    for name, control in training_controls.items()
}
torch.save(
    {
        "summaries": summaries,
        "training_controls": training_controls,
        "perplexity": curves,
        "run": run_info,
        "figure_arms": figure_arms,
    },
    run_dir / "overview.pt",
)
result_cache.write_manifest(run_dir)

## Reading the measurements

| Quantity | Definition |
| --- | --- |
| Distance to final state | For each physical core layer, mean over tokens of $\lVert h_{r,t}-h_{R,t}\rVert_2$, with $R=128$ by default. Each layer uses its own final recurrence, not a coda state. |
| Successive difference | Mean over tokens of $\lVert h_{r,t}-h_{r-1,t}\rVert_2$, starting at recurrence 2. |
| Attention distance to the Poisson training maximum | For each core layer, mean over heads of $\lVert A_{r,h}-A_{m,h}\rVert_F$, where $m=16$ by default. The Frobenius norm spans query and key positions, without sequence-length normalization; the reference uses the same layer, head, and example within the analysis rollout. |
| Sink rate | Fraction of heads whose mean attention to the first token is at least 0.3. Parcae adds no BOS, so this is the first text token. |
| Mixing score | Attention-row entropy divided by its causal maximum, averaged over query rows after the first, then over heads. |
| Column concentration | One minus normalized entropy of attention received across key positions; average over heads. |
| Residual norm | Mean of token-wise L2 norms of the raw block output, then an equal-weight mean across examples. This is not the norm of the mean token vector. |
| Residual entropy | Spectral entropy of the raw token-by-hidden residual matrix, in nats, matching stages of inference. |
| Perplexity | Exponential of total next-token NLL divided by the total predicted-token count, pooled over examples. |

Convergence and stage curves weight examples equally. Convergence bands show one
sample standard deviation; cyclic profiles omit recurrence bands for readability.
Norm-growth curves omit bands to keep six radii legible. No plot averages physical
core layers together except the final summary, which averages layers within
each recurrence and example before taking an equal-weight mean across examples.

The last observed state is a finite-rollout reference, not a demonstrated fixed
point: its plotted distance is zero by construction. Successive differences
provide the separate check that updates are vanishing. The radius dial controls
the diagonal injection, not the Jacobian of the complete nonlinear recurrent
update, so stability must be checked empirically rather than inferred from rho alone.

Fixed-depth training uses eight recurrences. Poisson training is clipped to 1–16;
the 128-recurrence analysis and the PPL points beyond those training ranges probe
extrapolation. Cyclic profiles separate recurrences into two rows: within the
training range (1–16 for Poisson, 1–8 for fixed) and beyond it. These one-based
counts describe visited depths; fixed training only stops at depth 8. All curves
are solid. The colour scale spans the complete rollout in both rows, while y-axis
limits are independent so late growth does not compress the within-training row.
Prelude appears with recurrence 1; the observed coda appears with the final
recurrence. Each row retains only its corresponding faint grey boundary connector.
Missing quick-mode arms retain both panels marked **skipped**; combined curves
and bars omit those arms.

In [ ]:
def rho_label(rho):
    value = r"\mathrm{learned}" if rho == "learned" else rho
    return rf"$\rho = {value}$"


def skipped(ax):
    ax.text(0.5, 0.5, "skipped", ha="center", va="center", transform=ax.transAxes)


def rho_subplots(schedule, *, columns=3):
    fig, axes = model_subplots(len(RHOS), columns=columns, height_per_subplot=1.6)
    for ax, name in zip(axes, figure_arms[schedule], strict=True):
        ax.set_title(rho_label(arm_parts(name)[1]))
    return fig, axes


def depth_positions(data, *, by_layer=False):
    positions = np.asarray(data["layers"] if by_layer else data["virtual_layers"])
    return positions / max(positions.max(), 1) * 100


def recurrence_groups(data):
    layers = np.asarray(data["layers"])
    core = data["recurrent_layers"]
    groups = {}
    if np.any(layers < min(core)):
        groups["Prelude"] = np.flatnonzero(layers < min(core))
    recurrences = np.asarray(data["recurrences"])
    for recurrence in sorted({r for r in recurrences if r is not None}):
        groups[f"R{recurrence}"] = np.flatnonzero(recurrences == recurrence)
    if np.any(layers > max(core)):
        groups["Coda"] = np.flatnonzero(layers > max(core))
    return groups


def draw_curve(ax, data, field, *, indices=None, x=None, band=True, **style):
    indices = np.arange(len(data["layers"])) if indices is None else indices
    x = np.asarray(data["virtual_layers"])[indices] if x is None else x
    mean = data["curves"][field]["mean"][indices]
    std = data["curves"][field]["std"][indices]
    (line,) = ax.plot(x, mean, marker="." if len(indices) == 1 else None, **style)
    if band:
        ax.fill_between(x, mean - std, mean + std, color=line.get_color(), alpha=0.2)
    return line


PROFILE_ROWS = ("Within train distribution", "Out of train distribution")


def label_profile_rows(axes):
    for row, label in zip(axes, PROFILE_ROWS, strict=True):
        row[0].annotate(
            label,
            xy=(0, 1),
            xycoords="axes fraction",
            xytext=(0, 20),
            textcoords="offset points",
            ha="left",
            va="bottom",
            weight="bold",
            annotation_clip=False,
        )


def draw_stage_profiles(ax, name, field, *, within_training):
    data = stages[name]
    schedule, _ = arm_parts(name)
    training_limit = (
        max_training_recurrence if schedule == "poisson" else training_recurrence
    )
    groups = recurrence_groups(data)
    n_recurrences = sum(key.startswith("R") for key in groups)
    # Recorded iterations start at zero; training limits count completed recurrences.
    visible = {
        key
        for key in groups
        if key.startswith("R")
        and ((int(key[1:]) + 1 <= training_limit) == within_training)
    }
    # Each boundary belongs only with its actual adjacent recurrence.
    if "R0" in visible:
        visible.add("Prelude")
    if f"R{n_recurrences - 1}" in visible:
        visible.add("Coda")
    x = depth_positions(data, by_layer=True)
    lines = {}
    for key, indices in groups.items():
        if key not in visible:
            continue
        if key in {"Prelude", "Coda"}:
            color, label = ("orange" if key == "Prelude" else "black"), key
        else:
            color = plt.colormaps[COLORMAPS["recurrences"]](
                int(key[1:]) / max(n_recurrences - 1, 1)
            )
            label = None
        lines[key] = draw_curve(
            ax,
            data,
            field,
            indices=indices,
            x=x[indices],
            band=False,
            color=color,
            label=label,
            linestyle="-",
            linewidth=1.25,
        )
    add_prelude_coda_connectors(ax, lines)
    ax.set(xlabel="Layer depth (%)", xticks=[0, 50, 100], xlim=(-5, 105))
    if field == "residual_norms_mean":
        ax.set_yscale("log")
    if not lines:
        ax.text(
            0.5, 0.5, "No recurrences", ha="center", va="center", transform=ax.transAxes
        )


def draw_mixing_extrapolation(ax, name):
    data = stages[name]
    groups = recurrence_groups(data)
    x = depth_positions(data, by_layer=True)
    late = [
        key
        for key in groups
        if key.startswith("R") and int(key[1:]) >= max_training_recurrence
    ]
    colors = plt.colormaps[COLORMAPS["recurrences"]](np.linspace(0, 1, len(late)))
    lines = {}
    for label, color in (("Prelude", "orange"), ("Coda", "black")):
        if label in groups:
            indices = groups[label]
            lines[label] = draw_curve(
                ax,
                data,
                "mixing_scores",
                indices=indices,
                x=x[indices],
                band=False,
                color=color,
                label=label,
            )
    for key, color in zip(late, colors, strict=True):
        indices = groups[key]
        lines[key] = draw_curve(
            ax,
            data,
            "mixing_scores",
            indices=indices,
            x=x[indices],
            band=False,
            color=color,
        )
    # The first visible recurrence is cropped: never link the prelude to it.
    add_prelude_coda_connectors(
        ax, {key: line for key, line in lines.items() if key != "Prelude"}
    )
    control = controls[name]
    draw_curve(
        ax,
        control,
        "mixing_scores",
        x=depth_positions(control, by_layer=True),
        band=False,
        color="red",
        linestyle="--",
        label="Control",
        zorder=3,
    )

In [ ]:
def rho_colors(n):
    return plt.colormaps["viridis"](np.linspace(0, 0.9, n))


def style_for(name):
    """Colour encodes spectral radius; line style encodes the sampler."""
    schedule, rho = arm_parts(name)
    colors = dict(zip(RHOS, rho_colors(len(RHOS)), strict=True))
    schedule_style = {
        "fixed": {"linestyle": "-", "marker": "o"},
        "poisson": {"linestyle": "--", "marker": "s"},
    }
    return {
        "color": colors[rho],
        **schedule_style[schedule],
        "markersize": 3,
    }


def rho_positions(names):
    """Place any arm subset on the shared spectral-radius axis."""
    positions = {rho: index for index, rho in enumerate(RHOS)}
    return np.asarray([positions[arm_parts(name)[1]] for name in names])


def clipped_poisson_distribution(mean, maximum):
    """Probability mass after clipping Poisson draws to [1, maximum]."""
    raw = [math.exp(-mean) * mean**k / math.factorial(k) for k in range(maximum)]
    probabilities = [sum(raw[:2]), *raw[2:maximum]]
    probabilities.append(1.0 - sum(probabilities))
    return np.arange(1, maximum + 1), np.asarray(probabilities)


def draw_perplexity(ax, names, *, yscale="log"):
    for name in names:
        _, rho = arm_parts(name)
        ax.plot(
            ppl_recurrences,
            [curves[name][t] for t in ppl_recurrences],
            label=rho_label(rho),
            **style_for(name),
        )
    ax.set(yscale=yscale, xlabel="Recurrences", ylabel="Perplexity")
    ax.set_xticks(ppl_recurrences, labels=[str(t) for t in ppl_recurrences])


def draw_poisson_training_distribution(ax):
    recurrences, probability = clipped_poisson_distribution(
        training_recurrence, max_training_recurrence
    )
    probability_ax = ax.twinx()
    probability_ax.bar(
        recurrences, probability, color="0.75", alpha=0.35, width=0.8, zorder=0
    )
    probability_ax.set(ylabel="Training probability", ylim=(0, probability.max() * 3))
    probability_ax.grid(False)
    probability_ax.set_zorder(ax.get_zorder() - 1)
    ax.patch.set_alpha(0)
    return probability_ax

In [ ]:
def plot_convergence(schedule, metric):
    fig, axes = rho_subplots(schedule, columns=2 if schedule == "fixed" else 3)
    for ax, name in zip(axes, figure_arms[schedule], strict=True):
        ax.set_xlabel("Recurrence")
        if ax.get_subplotspec().is_first_col():
            ax.set_ylabel(CONVERGENCE_METRICS[metric])
        if name not in stages:
            skipped(ax)
            continue
        data = stages[name]
        values = data["convergence"][metric]
        if metric == "attention_to_training_max":
            ax.axvline(
                max_training_recurrence, color="grey", linestyle="--", linewidth=0.8
            )
        first = 2 if metric == "successive" else 1
        x = np.arange(first, analysis_recurrences + 1)
        colors = plt.colormaps[COLORMAPS["layers"]](
            np.linspace(0, 1, len(data["recurrent_layers"]))
        )
        for average, spread, color in zip(
            values["mean"], values["std"], colors, strict=True
        ):
            ax.plot(x, average, color=color)
            ax.fill_between(
                x, average - spread, average + spread, color=color, alpha=0.2
            )
    add_depth_colorbar(fig, axes, "layer")
    save_figure(fig, run_dir, f"convergence/{metric}_{schedule}", bbox_inches=None)


def plot_attention_profiles(schedule, name):
    fig, axes = plt.subplots(
        2, 3, layout="constrained", figsize=get_figsize(True, (2, 3), 1.8)
    )
    for row, within_training in zip(axes, (True, False), strict=True):
        for ax, (field, label) in zip(row, ATTENTION_METRICS.items(), strict=True):
            ax.set_ylabel(label)
            if name in stages:
                draw_stage_profiles(ax, name, field, within_training=within_training)
            else:
                skipped(ax)
    label_profile_rows(axes)
    add_prelude_coda_legend(fig, include_control=False)
    fig.suptitle(
        f"140M · {schedule} · {rho_label(arm_parts(name)[1])}", x=0.01, ha="left"
    )
    add_depth_colorbar(fig, axes, "recurrence")
    rho_slug = arm_parts(name)[1].replace(".", "_")
    save_figure(
        fig, run_dir, f"recurrent/attention_{schedule}_rho_{rho_slug}", bbox_inches=None
    )


def plot_residual_profiles(schedule, field):
    names = figure_arms[schedule]
    for part, panel_names in enumerate((names[:3], names[3:]), start=1):
        columns = len(panel_names)
        fig, axes = plt.subplots(
            2,
            columns,
            layout="constrained",
            figsize=get_figsize(True, (2, columns), 1.8),
        )
        for row, within_training in zip(axes, (True, False), strict=True):
            for ax, name in zip(row, panel_names, strict=True):
                ax.set_title(rho_label(arm_parts(name)[1]))
                if name in stages:
                    draw_stage_profiles(
                        ax, name, field, within_training=within_training
                    )
                else:
                    skipped(ax)
                if ax.get_subplotspec().is_first_col():
                    ax.set_ylabel(RESIDUAL_METRICS[field])
        label_profile_rows(axes)
        add_prelude_coda_legend(fig, include_control=False)
        add_depth_colorbar(fig, axes, "recurrence")
        save_figure(
            fig, run_dir, f"recurrent/{field}_{schedule}_part_{part}", bbox_inches=None
        )


def plot_norm_growth():
    fig, axes = plt.subplots(
        1,
        2,
        sharex=True,
        sharey=True,
        layout="constrained",
        figsize=get_figsize(True, (1, 2), height_per_subplot=2.2),
    )
    handles = {}
    for ax, schedule in zip(axes, SCHEDULES, strict=True):
        for name in figure_arms[schedule]:
            if name not in stages:
                continue
            data = stages[name]
            rho = arm_parts(name)[1]
            line = draw_curve(
                ax,
                data,
                "residual_norms_mean",
                x=depth_positions(data),
                band=False,
                color=style_for(name)["color"],
                label=rho_label(rho),
            )
            handles[rho] = line
        ax.set(
            title=f"{schedule} sampler", xlabel="Virtual layer depth (%)", yscale="log"
        )
        if not any(name in stages for name in figure_arms[schedule]):
            skipped(ax)
    axes[0].set_ylabel("Mean residual norm")
    fig.legend(
        [handles[rho] for rho in RHOS if rho in handles],
        [rho_label(rho) for rho in RHOS if rho in handles],
        loc="outside upper center",
        ncols=len(handles),
    )
    save_figure(fig, run_dir, "residual_norm_growth", bbox_inches=None)


def plot_scaling_summary():
    fig, axes = plt.subplots(
        1, 2, layout="constrained", figsize=get_figsize(False, (1, 2), 1.5)
    )
    norm_ax = axes[0]
    for name in figure_arms["poisson"]:
        if name not in summaries:
            continue
        x, mean = recurrent_norm_by_recurrence(summaries[name])
        norm_ax.plot(
            x, mean, color=style_for(name)["color"], label=rho_label(arm_parts(name)[1])
        )
    norm_ax.set(
        title="Residual growth",
        xlabel="Recurrence",
        ylabel="Mean residual norm",
        yscale="log",
    )
    if not norm_ax.lines:
        skipped(norm_ax)
    norm_ax.axvline(max_training_recurrence, color="grey", linestyle=":", linewidth=0.8)

    ppl_ax = axes[1]
    names = [name for name in figure_arms["poisson"] if name in curves]
    draw_perplexity(ppl_ax, names, yscale="linear")
    probability_ax = draw_poisson_training_distribution(ppl_ax)
    probability_ax.set(ylim=(0, 0.2), yticks=[0, 0.1, 0.2])
    ppl_ax.set_title("Perplexity scaling")
    # Keep every measured point; fewer tick labels fit the narrower summary panel.
    ppl_ax.set_xticks(
        [
            t
            for t in ppl_recurrences
            if t == ppl_recurrences[0] or t >= training_recurrence
        ]
    )
    if not names:
        skipped(ppl_ax)
    handles = {}
    for ax in axes:
        for line, label in zip(*ax.get_legend_handles_labels(), strict=True):
            handles.setdefault(label, line)
    if handles:
        fig.legend(
            handles.values(),
            handles.keys(),
            loc="outside upper center",
            ncols=len(handles),
            columnspacing=0.6,
            handlelength=1.3,
            handletextpad=0.4,
        )
    save_figure(fig, run_dir, "stability_summary_scaling", bbox_inches=None)


def plot_mixing_summary():
    fig, axes = plt.subplots(
        1,
        2,
        sharex=True,
        sharey=True,
        layout="constrained",
        figsize=get_figsize(False, (1, 2), 1.5),
    )
    for ax, name in zip(axes, SUMMARY_ARMS, strict=True):
        ax.set(
            title=rho_label(arm_parts(name)[1]),
            xlabel="Layer depth (%)",
            xticks=[0, 50, 100],
            xlim=(-5, 105),
        )
        if name in stages:
            draw_mixing_extrapolation(ax, name)
        else:
            skipped(ax)
    axes[0].set_ylabel("Mixing score")
    add_prelude_coda_legend(fig)
    add_depth_colorbar(fig, axes, "recurrence", pad=0.05)
    save_figure(fig, run_dir, "stability_summary_mixing", bbox_inches=None)

## Approach to a fixed point

Each training scheme gets a final-reference figure and a successive-difference
figure. Fixed-scheme figures use three rows of two wider panels; Poisson uses
two rows of three. Colour identifies physical core layer, as in the convergence
notebook.

In [ ]:
plot_convergence("poisson", "to_final")

In [ ]:
plot_convergence("poisson", "successive")

In [ ]:
plot_convergence("fixed", "to_final")

In [ ]:
plot_convergence("fixed", "successive")

## Attention distance from the Poisson training maximum

For each Poisson-trained rho, compare every recurrent attention pattern with the
same layer and head at recurrence `max_training_recurrence` (16 by default) in the
128-recurrence analysis run. The reference index counts recurrences from one.

Compute a Frobenius norm over query and key positions **before** averaging heads;
then average examples equally. Coloured lines identify physical core layers,
bands show one sample standard deviation, and the grey marker locates the
reference recurrence. Distance is exactly zero there by construction. No norm is
divided by sequence length.

This uses the same six-panel layout as Poisson residual convergence. Omitted
quick-mode arms remain **skipped**. Fresh collection is required to obtain this
measurement: previously saved scalar attention statistics do not retain the
attention matrices needed to calculate it.

In [ ]:
plot_convergence("poisson", "attention_to_training_max")

## Attention metrics within each recurrence

One figure per radius and training scheme, with attention metrics in three
columns. The first row shows recurrences within the training range and the second
shows later recurrences. All six radii are always represented, including
placeholder figures for arms omitted in quick mode. There is no feedforward
control overlay.

In [ ]:
for name in figure_arms["poisson"]:
    plot_attention_profiles("poisson", name)

In [ ]:
for name in figure_arms["fixed"]:
    plot_attention_profiles("fixed", name)

## Residual metrics within each recurrence

Each metric and training scheme produces two figures of three rho columns:
0.25/0.5/0.75 in part 1, then 1.0/1.25/learned in part 2. The first row shows
recurrences within the training range and the second shows later recurrences.
All lines are solid, with the shared colourbar indexing early-to-late recurrences
across both rows. Feedforward controls are omitted.

In [ ]:
plot_residual_profiles("poisson", "residual_norms_mean")

In [ ]:
plot_residual_profiles("poisson", "residual_entropy")

In [ ]:
plot_residual_profiles("fixed", "residual_norms_mean")

In [ ]:
plot_residual_profiles("fixed", "residual_entropy")

## Residual norm growth through total depth

A single full-width figure places the training schemes side by side. Colour identifies
rho, and each curve shows the token-mean residual norm through the full execution,
including the prelude and coda. The shared logarithmic y-axis permits comparisons
between schemes without averaging their trajectories.

In [ ]:
plot_norm_growth()

## Perplexity against rollout length

The x-axis is linear, while perplexity remains logarithmic so unstable arms
remain legible. The left panel overlays the exact training distribution:
Poisson(8) clipped to 1--16 recurrences. The dashed line on the right marks
the fixed training depth of eight recurrences. A stable arm should be flat or
still improving beyond the depths seen in training.

The figures retain the original linear recurrence axis, logarithmic PPL scale,
training-distribution overlay, and degradation ratio at the longest measured
rollout relative to training depth. Colours and radius positions are fixed to the
complete sweep; omitted arms contribute no line or bar.

In [ ]:
fig, axes = plt.subplots(
    1,
    len(SCHEDULES),
    sharey=True,
    layout="constrained",
    figsize=get_figsize(True, (1, len(SCHEDULES)), height_per_subplot=2.2),
)
for ax, schedule in zip(np.atleast_1d(axes), SCHEDULES, strict=True):
    draw_perplexity(ax, [n for n in arms if arm_parts(n)[0] == schedule])
    if schedule == "poisson":
        draw_poisson_training_distribution(ax)
    else:
        ax.axvline(training_recurrence, color="grey", linewidth=1, linestyle="--")
        ax.text(
            training_recurrence,
            0.98,
            f" train = {training_recurrence}",
            color="grey",
            rotation=90,
            va="top",
            transform=ax.get_xaxis_transform(),
        )
    ax.set_title(f"{schedule} sampler")
# Gather by label without duplicating a radius shared by both samplers.
legend_handles = {}
for ax in np.atleast_1d(axes):
    for handle, label in zip(*ax.get_legend_handles_labels(), strict=True):
        legend_handles.setdefault(label, handle)
labels = [rho_label(rho) for rho in RHOS if rho_label(rho) in legend_handles]
fig.legend(
    [legend_handles[label] for label in labels],
    labels,
    loc="outside upper center",
    ncols=len(labels),
)
save_figure(fig, run_dir, "perplexity_vs_recurrence", bbox_inches=None)

In [ ]:
# How far each arm degrades beyond the depth it was trained at.
fig, ax = plt.subplots(
    figsize=get_figsize(False, (1, 1), height_per_subplot=1.8), layout="constrained"
)
longest = max(ppl_recurrences)
width = 0.8 / len(SCHEDULES)
for index, schedule in enumerate(SCHEDULES):
    names = [n for n in arms if arm_parts(n)[0] == schedule]
    ratios = [curves[n][longest] / curves[n][training_recurrence] for n in names]
    offset = (index - (len(SCHEDULES) - 1) / 2) * width
    ax.bar(
        rho_positions(names) + offset,
        ratios,
        width,
        label=f"{schedule} sampler",
    )
ax.axhline(1.0, color="grey", linewidth=0.8, linestyle=":")
ax.set_xticks(np.arange(len(RHOS)), labels=RHOS)
ax.set(
    xlabel=r"Injection spectral radius $\rho$",
    ylabel=f"Perplexity at {longest} over {training_recurrence}",
    yscale="log",
)
ax.legend()
save_figure(fig, run_dir, "perplexity_degradation", bbox_inches=None)

## Summary: stability and test-time scaling

Two separate, single-row figures use Poisson-trained models. Both export at
**3.25 × 1.5 inches**, ready to place side by side as paper subfigures without
rescaling the text.

**(a) Residual growth and perplexity scaling** (`stability_summary_scaling.pdf`)
shares a $\rho$ legend above the two panels. Residual growth is the mean token-wise
norm, averaged across recurrent layers within each recurrence and then equally
across examples. Prelude and coda are excluded. Its y-axis is logarithmic, the
training maximum is marked in grey, and curves omit uncertainty bands. Perplexity
retains linear axes and the clipped-Poisson training distribution behind the
curves; the probability axis spans **0.0–0.2**.

**(b) Mixing stages** (`stability_summary_mixing.pdf`) compares the two
`SUMMARY_ARMS` with shared x/y ranges and its own **Prelude / Coda / Control**
legend. Only recurrences beyond the training maximum (17–128 by default) are
coloured from early green to late yellow, on a shared physical-layer axis.
Orange and black show the prelude and coda after the full analysis rollout.
The dashed red control combines the prelude, final recurrence, and actual coda
from a separate 16-step run of the same model, examples, and initialization seed.
No line connects the prelude to the first displayed late recurrence; the final
recurrence retains its faint grey connector to the coda. These are raw
mixing-score profiles, with deviation visible relative to the control.

The two controls require only mixing scores and are cached separately from the
long-rollout analysis. Fresh collection shares their 16-step forwards with PPL
where examples overlap. Existing analysis caches remain reusable; only missing
controls need inference. Quick mode omits uncollected radius curves and labels
missing mixing panels **skipped**. Both figure cells reuse compact results.


In [ ]:
plot_scaling_summary()

In [ ]:
plot_mixing_summary()

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir, dataset=config.dataset)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")